# Single-Name CDS Pricing

A walk through `credit.SingleNameCDS` — the pricer for a vanilla single-name Credit Default Swap. We start from the deterministic-intensity model, build the two legs of the trade explicitly, derive the risky annuity (`RPV01`) and the par spread, mark a trade to market mid-life, and compute the standard first-order sensitivities (CS01, RR01, IR01) by bump-and-rebootstrap.

The companion notebook `05_credit_curve_bootstrapping.ipynb` covers how the survival curve $Q(t)$ used here is calibrated; this notebook focuses on what happens **after** the curve exists — i.e. the actual pricing of a CDS contract on a given pair of `(ZeroCurve, CreditCurve)`.

## Environment setup

The cell below calls `setup_demo_env()` from `examples/_setup.py`, which performs three steps:

1. Adds the project root to `sys.path` so first-party packages (`database`, `schedules`, `market_structures`, `credit`, ...) import without installation.
2. Redirects the global SQLite path to `examples/demo.db` via `set_db_path()`. The production `quant.db` is never read or written by this notebook.
3. Creates the domain tables and seeds the holidays table (USD/EUR/GBP/PLN, years 2000-2100) on first run; subsequent runs detect existing data and skip seeding.

A status line is printed when the cell runs so you can see which path was taken.

In [ ]:
import sys
sys.path.insert(0, "..")

from examples._setup import setup_demo_env
setup_demo_env()

## 1. The Pricing Model


The most common interpretation of $\lambda(t)$ is as a **hazard rate**. It is essential to distinguish between the intensity (a rate) and the probability (a dimensionless value).

### The Hazard Rate Definition
$\lambda(t)$ represents the **instantaneous arrival rate** of default. It is the probability that the reference entity will default in the next infinitesimal time step $dt$, given that it has survived up until time $t$. Mathematically, this is expressed as:

$$\lambda(t) dt \approx \mathbb{P}(\tau \in [t, t+dt] \mid \tau > t)$$

### Why multiply by $dt$?
*   **Dimensionality:** $\lambda(t)$ has the units of $1/\text{Time}$ (e.g., "per year"). To obtain a probability, which is dimensionless, you must multiply this rate by a time increment $dt$.
*   **The Speedometer Analogy:** If $\lambda(t)$ is the "speed" at which an entity is approaching default, $dt$ is the duration of travel. The probability of the "jump" occurring is the product of that speed and time.
*   **Values > 1:** Unlike a probability, the intensity $\lambda(t)$ can be greater than 1 (or 100%). For example, a highly distressed entity might have an intensity of 5.0, implying a massive "pressure" toward default, even though the probability of default over any interval can never exceed 1.0.

### Default time and survival probability

The default time $\tau$ of the reference name is modelled as the first jump of an inhomogeneous Poisson process with deterministic intensity $\lambda(t) \geq 0$. The two quantities the pricer needs are:

$$
Q(t) \;=\; \mathbb{P}(\tau > t) \;=\; \exp\!\Big(\!-\!\!\int_0^t \lambda(u)\,du\Big),
\qquad
-dQ(t) \;=\; \lambda(t)\,Q(t)\,dt
$$

$Q(t)$ is the **survival probability** to time $t$; $-dQ(t)$ is the risk-neutral probability mass of defaulting in $[t, t+dt]$. Both come from the `CreditCurve` via `non_default_probability(d)`.

### Derivation of the Default Probability Mass $-dQ(t)$

To understand the probability mass of defaulting in a specific time window, we derive $-dQ(t)$ from the survival probability $Q(t)$ using the **Chain Rule** and the **Fundamental Theorem of Calculus**.

**1. The Starting Expression**
We begin with the definition of survival probability:
$$Q(t) = \exp\left(-\int_0^t \lambda(u) du\right)$$

**2. Differentiate with Respect to Time**
We take the derivative $\frac{d}{dt}Q(t)$. According to the Chain Rule, $\frac{d}{dt}e^{g(t)} = e^{g(t)} \cdot g'(t)$. Here, $g(t) = -\int_0^t \lambda(u) du$.
$$\frac{dQ(t)}{dt} = \exp\left(-\int_0^t \lambda(u) du\right) \cdot \frac{d}{dt}\left( -\int_0^t \lambda(u) du \right)$$

**3. Apply the Fundamental Theorem of Calculus**
The derivative of the integral $\int_0^t \lambda(u) du$ is simply the integrand $\lambda(t)$. Therefore:
$$\frac{dQ(t)}{dt} = Q(t) \cdot (-\lambda(t))$$

**4. Final Form**
Rearranging the terms to isolate the infinitesimal change, we get:
$$-dQ(t) = \lambda(t) Q(t) dt$$

**Interpretation:**
*   $Q(t)$ is the probability of surviving until time $t$.
*   $\lambda(t)dt$ is the conditional probability of defaulting in the next instant.
*   $-dQ(t)$ is the **unconditional probability** (the "mass") of the first jump occurring in the interval $[t, t+dt]$. 
*   The negative sign is necessary because $Q(t)$ is a decreasing function; a loss in survival probability represents a gain in default probability.

### The CDS contract

A vanilla single-name CDS exchanges two streams of cash flows between a **protection buyer** and a **protection seller**:

| Leg | Paid by | Cash flow | Frequency |
|---|---|---|---|
| Premium leg | Buyer | running coupon $s\cdot \alpha_i \cdot N$ on each accrual end $T_i$, **conditional on no default** | quarterly (typically) |
| Premium accrual | Buyer | accrued coupon $s\cdot \alpha(T_{i-1},\tau)\cdot N$ at $\tau$ | one-off, on default |
| Protection leg | Seller | $(1-R)\cdot N$ at the default time $\tau$ | one-off, on default |

where $s$ is the contractual running spread, $\alpha_i$ is the accrual day-count fraction over period $(T_{i-1}, T_i]$, $N$ is the trade notional, and $R$ is the (deterministic) recovery rate.

### Closed-form leg PVs

Discounting by the risk-free $P(t)$, integrating against the default density, and accruing only over surviving paths gives:

$$
\text{Premium running PV}
\;=\; s\,N \sum_i \alpha_i\, P(T_i^{\text{pay}})\, Q(T_i)
$$

$$
\text{Premium AoD PV}
\;=\; s\,N \sum_i \int_{T_{i-1}}^{T_i} \!\alpha(T_{i-1},t)\, P(t)\,\lambda(t)\,Q(t)\,dt
$$

$$
\text{Protection PV}
\;=\; (1-R)\,N \sum_i \int_{T_{i-1}}^{T_i}\! P(t)\,\lambda(t)\,Q(t)\,dt
$$

### Midpoint approximation

`SingleNameCDS` evaluates the two integrals period-by-period using the **midpoint discount factor** and the exact default-probability increment $\Delta Q_i = Q(T_{i-1}) - Q(T_i)$:

$$
\int_{T_{i-1}}^{T_i} P(t)\,\lambda(t)\,Q(t)\,dt
\;\approx\;
\underbrace{\tfrac{1}{2}\big(P(T_{i-1}) + P(T_i)\big)}_{\text{DF}_{\text{mid},i}}\cdot
\underbrace{\big(Q(T_{i-1}) - Q(T_i)\big)}_{\Delta Q_i}
$$

For the accrual-on-default integral, $\alpha(T_{i-1},t)$ is replaced by its midpoint $\alpha_i/2$. This is exact in the limit of vanishing accrual length, vanishes correctly at zero hazard rate, and is the standard ISDA-style formula. Equivalent integrals are sometimes evaluated assuming piecewise-constant hazard between premium dates — the midpoint variant is simpler, has the same first-order accuracy, and is what the implementation uses.

### Par spread and RPV01

The **risky annuity** (or **RPV01**) is the premium-leg PV per unit running spread per unit notional:

$$
\text{RPV01}
\;=\;
\sum_i \alpha_i\, P(T_i^{\text{pay}})\, Q(T_i)
\;+\;
\sum_i \tfrac{\alpha_i}{2}\,\text{DF}_{\text{mid},i}\,\Delta Q_i
$$

The **par spread** is the running spread that makes the trade NPV zero at inception:

$$
s^* \;=\; \frac{\text{Protection PV}}{\text{RPV01}\cdot N}
$$

This is the spread the bootstrapper drives each calibrating quote toward. For an off-market trade priced after the curve has moved, the buyer's NPV satisfies, to first order,

$$
\text{NPV}_{\text{buyer}} \;\approx\; (s^* - s_{\text{contract}})\cdot \text{RPV01}\cdot N.
$$

In [ ]:
import math
from datetime import date, timedelta

import matplotlib.pyplot as plt

from market_conventions import BusinessDayConvention, DayCountConvention, StubType
from market_structures.rates.curve import ZeroCurve
from schedules import CalendarType, Frequency

from credit import (
    CdsQuote,
    CdsSide,
    CreditCurveBootstrapper,
    InterpolationVariable,
    SingleNameCDS,
)

REF      = date(2024, 1, 2)
USD      = CalendarType.USD
MF       = BusinessDayConvention.FOLLOWING
ACT360   = DayCountConvention.ACT_360
ACT365   = DayCountConvention.ACT_365_FIXED
RECOVERY = 0.40
NOTIONAL = 10_000_000.0

print(f"Reference (pricing) date  : {REF}")
print(f"Recovery rate (constant)  : {RECOVERY:.0%}")
print(f"Trade notional            : ${NOTIONAL:,.0f}")

## 2. Building the Pricing Inputs

`SingleNameCDS` is curve-agnostic: it consumes a `ZeroCurve` for risk-free discounting and a `CreditCurve` for survival probabilities. Both must share the pricing date. We use:

- **`ZeroCurve`** — a small term structure of continuously-compounded zero rates (4.5% / 4.2% / 4.0% at 2Y / 5Y / 10Y) on `ACT/365F`.
- **`CreditCurve`** — bootstrapped from a five-tenor IG-style USD CDS term structure (80 / 110 / 130 / 145 / 155 bps at 1Y / 3Y / 5Y / 7Y / 10Y) with `FORWARD_DEFAULT_SPREAD` interpolation (piecewise-constant hazard rates, ISDA-style).

The credit-curve bootstrap is covered in detail in notebook 05; we just call it here.

In [ ]:
zero_curve = ZeroCurve(
    reference_date=REF,
    pillar_dates=[date(2026, 1, 2), date(2029, 1, 2), date(2034, 1, 2)],
    rates=[0.045, 0.042, 0.040],
    day_count_convention=ACT365,
)

spreads_bps = {"1Y": 80, "3Y": 110, "5Y": 130, "7Y": 145, "10Y": 155}

def make_quote(tenor: str, bps: float) -> CdsQuote:
    return CdsQuote(
        spread=bps / 10_000.0,
        tenor=tenor,
        pay_frequency=Frequency.QUARTERLY,
        calendar=USD,
        business_day_convention=MF,
        day_count_convention=ACT360,
        stub_type=StubType.SHORT_FRONT,
    )

market_quotes = [make_quote(t, bps) for t, bps in spreads_bps.items()]

credit_curve = CreditCurveBootstrapper(
    reference_date=REF,
    quotes=market_quotes,
    zero_curve=zero_curve,
    recovery_rate=RECOVERY,
    interpolation_variable=InterpolationVariable.FORWARD_DEFAULT_SPREAD,
).bootstrap()

print(credit_curve.summary())

**Reading `CreditCurve.summary()`** — one row per pillar, columns:

| Column | Meaning |
|---|---|
| `Pillar` | Pillar date. |
| `Tenor` | Year fraction $t_i$ from the reference date, under the curve's day-count convention. |
| `Q` | Survival probability $Q(t_i) = \mathbb{P}(\tau > t_i)$. |
| `DefSpread` | **Cumulative-equivalent default spread** $s(t_i) = -\ln Q(t_i)/t_i$ — anchored at the reference date. Equivalently, the time-average of the hazard rate from $0$ to $t_i$. |
| `FwdHazard (lambda(t))` | **Segment-local forward hazard** $-\ln(Q_i/Q_{i-1})/(t_i - t_{i-1})$ from the previous pillar. Under `FORWARD_DEFAULT_SPREAD` interpolation this is the piecewise-constant $\lambda$ on the segment exactly. |

For pillar 1 the two spread columns coincide; further out they diverge whenever the term structure of hazards is non-flat. Reading `DefSpread` across pillars tells you the cumulative average; reading `FwdHazard` tells you the per-segment forward.

## 3. Constructing the Pricer

`SingleNameCDS(...)` takes the following parameters:

| Parameter | Meaning |
|---|---|
| `pricing_date` | Valuation date $t_v$. Must equal the `reference_date` of both curves. |
| `schedule` | Premium-leg `Schedule` (accrual-start, accrual-end, pay-date, day-count fraction per period). For a new trade pass `quote.schedule(reference_date)`. For mid-life, build a schedule from the last coupon date to maturity. |
| `spread` | Contractual running coupon $s$ in decimal (`0.0130` = 130 bps). Fixed for the life of the trade. |
| `recovery_rate` | Constant $R \in [0,1)$ paid at default. |
| `zero_curve` | Risk-free curve providing $P(t)$. |
| `credit_curve` | Survival curve providing $Q(t)$. |
| `notional` | Trade notional $N$. Defaults to 1. |
| `side` | `CdsSide.BUYER` (default) or `CdsSide.SELLER` — sign convention only. |

Internally the constructor:

1. Validates the curve reference dates match `pricing_date`.
2. Calls `schedule.generate()` to expand the schedule into `Period` objects.
3. **Drops periods that are fully elapsed** (`accrual_end <= pricing_date`); the live periods retain their original `dcf` because the contract pays $s\,\alpha_i\,N$ regardless of when valuation occurs mid-period.

We instantiate the 5Y / 130 bps trade as our running example.

In [ ]:
quote_5y    = market_quotes[2]              # the 5Y / 130 bps quote
schedule_5y = quote_5y.schedule(REF)

cds = SingleNameCDS(
    pricing_date=REF,
    schedule=schedule_5y,
    spread=quote_5y.quote_value(),
    recovery_rate=RECOVERY,
    zero_curve=zero_curve,
    credit_curve=credit_curve,
    notional=NOTIONAL,
    side=CdsSide.BUYER,
)

periods = list(schedule_5y)
print(f"Trade tenor              : {quote_5y.tenor}")
print(f"Contractual spread       : {quote_5y.quote_value()*10_000:.2f} bps")
print(f"Premium accrual periods  : {len(periods)}")
print(f"First period             : {periods[0].accrual_start} -> {periods[0].accrual_end}  (DCF {periods[0].dcf:.4f})")
print(f"Last period              : {periods[-1].accrual_start} -> {periods[-1].accrual_end}  (DCF {periods[-1].dcf:.4f})")

## 4. Premium Leg

The premium leg has two pieces. The **running coupon** is paid on each accrual end conditional on no default by then:

$$
\text{Premium running PV}
\;=\; s\,N \sum_i \alpha_i\, P(T_i^{\text{pay}})\, Q(T_i)
$$

The **accrual on default** captures the partial coupon paid out at $\tau$ when default falls inside an accrual period — using the midpoint approximation:

$$
\text{Premium AoD PV}
\;\approx\; s\,N \sum_i \tfrac{\alpha_i}{2}\,\text{DF}_{\text{mid},i}\,\Delta Q_i
$$

The total premium leg PV is the sum of the two. Per-period detail is exposed by `premium_leg_summary()`, which prints the building blocks $\alpha_i$, $Q(T_i)$, $P(T_i^{\text{pay}})$, the undiscounted cash flow $s\alpha_i N$, and both PV components per period.

In [ ]:
running = cds.premium_leg_running_pv()
aod     = cds.accrual_on_default_pv()
premium = cds.premium_leg_pv()

print(f"Running coupons PV       : ${running:>14,.2f}")
print(f"Accrual-on-default PV    : ${aod:>14,.2f}")
print(f"Premium leg PV (total)   : ${premium:>14,.2f}")
print()
print(f"AoD as % of running      : {aod/running:>13.2%}")
print(f"AoD as % of total premium: {aod/premium:>13.2%}")

In [ ]:
print(cds.premium_leg_summary())

## 5. Protection Leg

The protection leg pays the loss-given-default $1-R$ at the default time $\tau$ if it occurs within the contract life. Period-by-period:

$$
\text{Protection PV}
\;\approx\; (1-R)\,N \sum_i \text{DF}_{\text{mid},i}\,\Delta Q_i
$$

Each period contribution is the loss-given-default times the **conditional default probability mass** $\Delta Q_i$ in that period, discounted by the midpoint discount factor. The dominant drivers are the period default probabilities — $\Delta Q_i$ is small for short-dated, low-spread names and grows with both spread level and tenor. `protection_leg_summary()` prints the per-period breakdown ($Q_{i-1}$, $Q_i$, $\Delta Q_i$, $\text{DF}_{\text{mid},i}$, and the dollar contribution to the protection PV).

In [ ]:
prot = cds.protection_leg_pv()
print(f"Protection leg PV : ${prot:>14,.2f}")
print()
print(cds.protection_leg_summary())

## 6. RPV01 and the Par Spread

The **risky annuity** (a.k.a. RPV01, the credit analogue of DV01) is the premium leg PV per unit running spread per unit notional — i.e. exactly the same sum that defines `premium_leg_pv` divided out by $s$ and $N$:

$$
\text{RPV01} \;=\; \frac{\text{Premium leg PV}}{s\, N}
$$

It captures everything in the premium leg that does **not** depend on the contractual spread, and is the natural denominator for converting an NPV move into a spread move.

The **par spread** $s^*$ is the value of $s$ that makes the trade NPV zero on the current curves:

$$
s^* \;=\; \frac{\text{Protection PV}}{\text{RPV01}\cdot N}
$$

If the credit curve was bootstrapped from this exact tenor, `par_spread()` should equal the input quote to within the bootstrapper's tolerance — that round-trip is the bootstrap's defining property.

In [ ]:
rpv01 = cds.rpv01()
par   = cds.par_spread()

print(f"RPV01 (premium PV / (spread * notional))      : {rpv01 / NOTIONAL:.6f}  per unit notional")
print(f"  -> dollar RPV01                             : ${rpv01:,.2f}")
print(f"Par spread implied by the curves              : {par*10_000:.4f} bps")
print(f"Market input spread (5Y quote)                : {quote_5y.quote_value()*10_000:.4f} bps")
print()

# Sanity: at par the trade has zero NPV up to bootstrap tolerance.
cds_at_par = SingleNameCDS(
    pricing_date=REF, schedule=schedule_5y, spread=par,
    recovery_rate=RECOVERY, zero_curve=zero_curve, credit_curve=credit_curve,
    notional=NOTIONAL, side=CdsSide.BUYER,
)
print(f"NPV at par spread                             : {cds_at_par.npv():.4e}")

## 7. NPV, Sides, and Off-Market Mark-to-Market

The NPV decomposes as

$$
\text{NPV}_{\text{buyer}} \;=\; \text{Protection PV} \;-\; \text{Premium leg PV},
\qquad
\text{NPV}_{\text{seller}} \;=\; -\,\text{NPV}_{\text{buyer}}.
$$

`CdsSide` is a **reporting** convention, not a pricing knob — both sides see the same legs; only the sign flips. NPV is not zero in general: it is zero by construction when $s = s^*$, and otherwise

$$
\text{NPV}_{\text{buyer}} \;\approx\; (s^* - s)\cdot \text{RPV01}\cdot N \;+\; \mathcal{O}\!\big((s^*-s)^2\big).
$$

This is the daily P&L pattern of a CDS book: when the credit curve widens, the par spread rises above the contractual spread and the long-protection (buyer) book's NPV moves up by approximately the spread change times RPV01.

In [ ]:
buyer  = SingleNameCDS(pricing_date=REF, schedule=schedule_5y, spread=quote_5y.quote_value(),
                       recovery_rate=RECOVERY, zero_curve=zero_curve, credit_curve=credit_curve,
                       notional=NOTIONAL, side=CdsSide.BUYER)
seller = SingleNameCDS(pricing_date=REF, schedule=schedule_5y, spread=quote_5y.quote_value(),
                       recovery_rate=RECOVERY, zero_curve=zero_curve, credit_curve=credit_curve,
                       notional=NOTIONAL, side=CdsSide.SELLER)

print(f"At-market 5Y trade  ({quote_5y.quote_value()*10_000:.0f} bps)")
print(f"  Buyer NPV  : ${buyer.npv():>+14,.2f}")
print(f"  Seller NPV : ${seller.npv():>+14,.2f}")
print()

# Off-market: a trade entered earlier at 100 bps, now valued against the current market.
contract_spread = 0.0100
off_mkt_buyer = SingleNameCDS(
    pricing_date=REF, schedule=schedule_5y, spread=contract_spread,
    recovery_rate=RECOVERY, zero_curve=zero_curve, credit_curve=credit_curve,
    notional=NOTIONAL, side=CdsSide.BUYER,
)
linear_estimate = (par - contract_spread) * rpv01

print(f"Off-market: 5Y bought at {contract_spread*10_000:.0f} bps; market par is now {par*10_000:.2f} bps")
print(f"  Exact buyer NPV                       : ${off_mkt_buyer.npv():>+14,.2f}")
print(f"  Linear approx (s* - s) * RPV01 * N    : ${linear_estimate:>+14,.2f}")
print(f"  Difference (second-order)             : ${off_mkt_buyer.npv() - linear_estimate:>+14,.2f}")

## 8. Mid-Life Pricing

Trades do not stay at-market forever. Six months after inception we revalue the **same** 5Y trade against the curves observed at that later date. The pricer is built to handle this transparently:

- `pricing_date` is moved forward to the new valuation date.
- The **same** schedule (originally generated at trade inception) is passed in. Periods whose `accrual_end` is at or before the new pricing date are dropped automatically by `_clip_periods`.
- The first live period retains its **full original DCF**: the running coupon for that period is contractually fixed for the whole accrual period regardless of when it is valued. Only the protection-leg and accrual-on-default integrals are clipped to begin at the pricing date (via `_integration_start` inside the pricer).

Curves observed at the later date are passed in fresh; the bootstrap below uses the same five market quotes (i.e. it assumes the term structure of par spreads has not moved), so any NPV change is purely from the passage of time.

In [ ]:
later_date = date(2024, 7, 2)

zero_curve_later = ZeroCurve(
    reference_date=later_date,
    pillar_dates=[date(2026, 7, 2), date(2029, 7, 2), date(2034, 7, 2)],
    rates=[0.045, 0.042, 0.040],
    day_count_convention=ACT365,
)
credit_curve_later = CreditCurveBootstrapper(
    reference_date=later_date,
    quotes=market_quotes,
    zero_curve=zero_curve_later,
    recovery_rate=RECOVERY,
    interpolation_variable=InterpolationVariable.FORWARD_DEFAULT_SPREAD,
).bootstrap()

mid_life = SingleNameCDS(
    pricing_date=later_date,
    schedule=schedule_5y,                              # unchanged from inception
    spread=quote_5y.quote_value(),
    recovery_rate=RECOVERY,
    zero_curve=zero_curve_later,
    credit_curve=credit_curve_later,
    notional=NOTIONAL,
    side=CdsSide.BUYER,
)

remaining = sum(1 for p in schedule_5y if p.accrual_end > later_date)
total     = len(list(schedule_5y))
print(f"Pricing date            : {later_date}  (6 months after {REF})")
print(f"Periods remaining       : {remaining}/{total}  (already-paid premium dates have been dropped)")
print()
print(f"Inception NPV (buyer)   : ${cds.npv():>+14,.2f}")
print(f"Mid-life NPV (buyer)    : ${mid_life.npv():>+14,.2f}")
print(f"NPV change in 6 months  : ${mid_life.npv() - cds.npv():>+14,.2f}")

## 9. First-Order Sensitivities

Three first-order sensitivities are reported on every CDS book. None of them is a closed-form derivative — each is a finite-difference computed by **bumping a market input, re-bootstrapping the credit curve, and re-pricing**. The bump-and-rebootstrap is essential: $Q(t)$ is a function of all market inputs, so naive differentiation through the closed-form pricer with the curve held fixed gives the wrong answer.

| Sensitivity | Bumped input | Bump size | Interpretation |
|---|---|---|---|
| **CS01** (parallel) | All CDS spreads | +1 bp uniform | Change in NPV per parallel 1 bp widening of the credit curve. |
| **Bucket CS01** | One pillar's CDS spread | +1 bp | Key-rate sensitivity to a specific pillar; sums (approximately) to the parallel CS01. |
| **RR01** | Recovery rate $R$ | +1% | Sensitivity to the recovery assumption (sticky-spread re-bootstrap — assumes market quotes are unchanged when $R$ moves). |
| **IR01** | Risk-free curve rates | +1 bp parallel | Sensitivity to the discount curve only. |

A protection **buyer** has positive CS01: the book gains when credit spreads widen, because the par spread rises above the contractual spread and existing protection is now bought "cheap". The sign of IR01 depends on the trade direction and remaining maturity but is generally an order of magnitude smaller than CS01.

We define a small helper `reprice(...)` that wraps the bump-and-rebootstrap loop, then use it to compute parallel CS01, per-pillar bucket CS01s, RR01, and IR01.

In [ ]:
def reprice(spread_bumps_bps=None, rr=RECOVERY, zero=zero_curve):
    """Re-bootstrap the credit curve under perturbed inputs and return buyer NPV."""
    quotes = (
        market_quotes if spread_bumps_bps is None
        else [q.bumped(b / 10_000.0) for q, b in zip(market_quotes, spread_bumps_bps)]
    )
    cc = CreditCurveBootstrapper(
        reference_date=REF, quotes=quotes, zero_curve=zero, recovery_rate=rr,
        interpolation_variable=InterpolationVariable.FORWARD_DEFAULT_SPREAD,
    ).bootstrap()
    pricer = SingleNameCDS(
        pricing_date=REF, schedule=schedule_5y, spread=quote_5y.quote_value(),
        recovery_rate=rr, zero_curve=zero, credit_curve=cc,
        notional=NOTIONAL, side=CdsSide.BUYER,
    )
    return pricer.npv()

base_npv      = reprice()
parallel_npv  = reprice(spread_bumps_bps=[1] * len(market_quotes))
parallel_cs01 = parallel_npv - base_npv

print(f"Base NPV                            : ${base_npv:>+14,.2f}")
print(f"NPV after +1 bp parallel CDS bump   : ${parallel_npv:>+14,.2f}")
print(f"Parallel CS01                       : ${parallel_cs01:>+14,.2f}")
print()

print(f"{'Tenor':<6} {'Bucket CS01 ($)':>18}")
print("-" * 25)
bucket_total = 0.0
for i, tenor in enumerate(spreads_bps):
    bumps = [0] * len(market_quotes); bumps[i] = 1
    bucket = reprice(spread_bumps_bps=bumps) - base_npv
    bucket_total += bucket
    print(f"{tenor:<6} {bucket:>18,.2f}")
print("-" * 25)
print(f"{'Sum':<6} {bucket_total:>18,.2f}     (~ parallel CS01 to first order)")

In [ ]:
rr_up_npv = reprice(rr=RECOVERY + 0.01)
rr01 = rr_up_npv - base_npv

print(f"NPV at recovery {RECOVERY:.0%}     : ${base_npv:>+14,.2f}")
print(f"NPV at recovery {RECOVERY+0.01:.0%}     : ${rr_up_npv:>+14,.2f}")
print(f"RR01 (per +1% recovery): ${rr01:>+14,.2f}")
print()
print("This is a 'sticky-spread' RR01 — market quotes are held fixed and the curve is")
print("re-bootstrapped under the new R. Two effects partially cancel:")
print("  (i)  protection payout (1 - R) shrinks            -> protection PV falls;")
print("  (ii) implied hazard rises (lambda ~ s/(1-R))      -> Q(t) drops faster.")
print("Net RR01 is small for IG names; can be material for distressed credits.")

In [ ]:
zero_curve_up = ZeroCurve(
    reference_date=REF,
    pillar_dates=[date(2026, 1, 2), date(2029, 1, 2), date(2034, 1, 2)],
    rates=[0.045 + 1e-4, 0.042 + 1e-4, 0.040 + 1e-4],
    day_count_convention=ACT365,
)
ir_up_npv = reprice(zero=zero_curve_up)
ir01 = ir_up_npv - base_npv

print(f"NPV with discount curve unchanged     : ${base_npv:>+14,.2f}")
print(f"NPV with discount curve +1 bp parallel: ${ir_up_npv:>+14,.2f}")
print(f"IR01                                  : ${ir01:>+14,.2f}")
print()
print("IR01 is generally an order of magnitude smaller than CS01 for short-/medium-tenor")
print("CDS — the trade is a credit instrument, not a rates instrument. It is not zero")
print("because both legs are present-valued through P(t), but the protection-leg DV01")
print("partially offsets the premium-leg DV01.")

In [ ]:
sample_dates = [REF + timedelta(days=d) for d in range(7, 365 * 10 + 1, 7)]
t_axis = [(d - REF).days / 365.0 for d in sample_dates]

q_curve  = [credit_curve.non_default_probability(d) for d in sample_dates]
df_curve = [zero_curve.discount_factor(d) for d in sample_dates]

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.2))

axes[0].plot(t_axis, q_curve, color='steelblue', linewidth=2)
axes[0].set_title('Survival probability $Q(t)$')
axes[0].set_xlabel('Years'); axes[0].set_ylabel('Q(t)')
axes[0].grid(alpha=0.3)

axes[1].plot(t_axis, df_curve, color='tomato', linewidth=2)
axes[1].set_title('Risk-free discount factor $P(t)$')
axes[1].set_xlabel('Years'); axes[1].set_ylabel('P(t)')
axes[1].grid(alpha=0.3)

# Buyer NPV as a function of the contractual spread, all else equal.
spread_grid = [s / 10_000.0 for s in range(50, 251, 5)]
npv_curve = []
for s in spread_grid:
    pricer = SingleNameCDS(
        pricing_date=REF, schedule=schedule_5y, spread=s,
        recovery_rate=RECOVERY, zero_curve=zero_curve, credit_curve=credit_curve,
        notional=NOTIONAL, side=CdsSide.BUYER,
    )
    npv_curve.append(pricer.npv())

axes[2].plot([s * 10_000 for s in spread_grid], [v / 1e6 for v in npv_curve],
             color='mediumpurple', linewidth=2)
axes[2].axhline(0, color='black', linewidth=0.7)
axes[2].axvline(par * 10_000, color='gray', linestyle='--', alpha=0.6,
                label=f'par spread = {par*10_000:.1f} bps')
axes[2].set_title('Buyer NPV vs contractual spread (curves fixed)')
axes[2].set_xlabel('Contractual spread (bps)')
axes[2].set_ylabel('NPV ($ millions)')
axes[2].grid(alpha=0.3); axes[2].legend()

plt.tight_layout()
plt.show()

## Summary

`SingleNameCDS` decomposes a CDS trade into a premium leg (running coupons plus accrual on default) and a protection leg (contingent recovery payment), each evaluated period-by-period with the midpoint discount-factor approximation. The risky annuity (`rpv01`) and the par spread (`par_spread`) characterise the curve-implied break-even spread; an off-market trade's NPV is captured to first order by $(s^* - s_{\text{contract}}) \cdot \text{RPV01} \cdot N$. The same pricer handles new and mid-life trades — periods fully before `pricing_date` are dropped automatically. First-order sensitivities (CS01, RR01, IR01) are computed by **bump-and-rebootstrap**: each shock to a market input requires re-calibrating the credit curve before re-pricing, because the survival curve is itself a derived quantity.